# День 7 — Анализ ошибок и мини-приложение

**Цель:** разобраться, на чём именно ошибается модель, и собрать демо.

Переиспользуемый код — в [`error_analysis.py`](error_analysis.py) и [`predictor.py`](predictor.py).

In [1]:
from compare_utils import (
    get_test_split, load_fine_tuned, load_baseline, load_tfidf,
    predict_fine_tuned, predict_baseline, predict_tfidf,
)
from tokenization_utils import load_tokenizer
from embeddings_utils import load_model

test_texts, test_labels = get_test_split()

model_ft, tokenizer = load_fine_tuned()
preds_ft = predict_fine_tuned(test_texts, model_ft, tokenizer)
y_pred_ft = [p['prediction'] for p in preds_ft]
probs_ft = [p['probabilities'] for p in preds_ft]

print(f'Тестовая выборка: {len(test_texts)}')

C:\Users\Vsevolod\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Тестовая выборка: 600


## Задача 1: Анализ False Positive и False Negative

In [2]:
import pandas as pd

df_test = pd.DataFrame({
    'text': test_texts,
    'true_label': test_labels,
    'pred_label': y_pred_ft,
    'confidence': [p.max() for p in probs_ft],
})

errors = df_test[df_test['true_label'] != df_test['pred_label']]

# False Positives: предсказали positive (1), а было negative (0)
fp = errors[(errors['pred_label'] == 1) & (errors['true_label'] == 0)]
# False Negatives: предсказали negative (0), а было positive (1)
fn = errors[(errors['pred_label'] == 0) & (errors['true_label'] == 1)]

print(f'Всего ошибок: {len(errors)} из {len(df_test)} ({len(errors)/len(df_test):.1%})')
print(f'False Positives: {len(fp)}')
print(f'False Negatives: {len(fn)}')

Всего ошибок: 53 из 600 (8.8%)
False Positives: 24
False Negatives: 29


## Задача 2: Анализ паттернов ошибок

In [3]:
print("=== FALSE POSITIVES (сказали good, а было bad) ===")
for _, row in fp.head(5).iterrows():
    print(f'\nТекст: {row["text"][:100]}')
    print(f'Истинный: negative, Предсказан: positive (уверенность {row["confidence"]:.3f})')

print("\n\n=== FALSE NEGATIVES (сказали bad, а было good) ===")
for _, row in fn.head(5).iterrows():
    print(f'\nТекст: {row["text"][:100]}')
    print(f'Истинный: positive, Предсказан: negative (уверенность {row["confidence"]:.3f})')

=== FALSE POSITIVES (сказали good, а было bad) ===

Текст: there is plenty of room for editing , and
Истинный: negative, Предсказан: positive (уверенность 0.778)

Текст: bargain-basement
Истинный: negative, Предсказан: positive (уверенность 0.747)

Текст: fit all of pootie tang in between its punchlines
Истинный: negative, Предсказан: positive (уверенность 0.886)

Текст: stifling morality tale
Истинный: negative, Предсказан: positive (уверенность 0.931)

Текст: restage the whole thing
Истинный: negative, Предсказан: positive (уверенность 0.579)


=== FALSE NEGATIVES (сказали bad, а было good) ===

Текст: it will do so in a way that does n't make you feel like a sucker
Истинный: positive, Предсказан: negative (уверенность 0.644)

Текст: that rival vintage looney tunes for the most creative mayhem in a brief amount of time
Истинный: positive, Предсказан: negative (уверенность 0.595)

Текст: that accomplishes so much that one viewing ca n't possibly be enough
Истинный: positive, Предсказа

### Зависит ли ошибка от длины текста?

Задание предполагает, что да. Проверим это, а не примем на веру.

In [4]:
from error_analysis import build_error_frame, analyze_errors, length_buckets

df = build_error_frame(test_texts, test_labels, y_pred_ft, probs_ft)
stats = analyze_errors(df)

print(f'Средняя длина ошибочных текстов: {stats["mean_len_errors"]:.0f} симв.')
print(f'Средняя длина всех текстов:      {stats["mean_len_all"]:.0f} симв.')
print()
print('Доля ошибок по длине (в словах):')
print(length_buckets(df))

Средняя длина ошибочных текстов: 53 симв.
Средняя длина всех текстов:      52 симв.

Доля ошибок по длине (в словах):
           n  errors  error_rate
n_words                         
0-5      230      20    0.086957
5-10     148      14    0.094595
10-15     96       6    0.062500
15-25     85       8    0.094118
25-100    41       5    0.121951


**Вывод против ожидания:** явной зависимости нет. Доля ошибок скачет между 6.2% и 12.2% без монотонного тренда — самые короткие тексты (0–5 слов) ошибаются в 8.7% случаев, а фразы 10–15 слов даже реже (6.2%). При 53 ошибках на 5 корзин в каждой оказывается по 5–20 штук, и такие колебания — обычный шум.

Длина текста здесь **не** объясняет ошибки. Ищем дальше.

### Уверенность на ошибках — вот это интереснее

In [5]:
print(f'Средняя уверенность на ВЕРНЫХ:    {stats["conf_correct"]:.3f}')
print(f'Средняя уверенность на ОШИБОЧНЫХ: {stats["conf_errors"]:.3f}')
print(f'\nОшибок с уверенностью > 0.9: {stats["n_confident_errors"]} из {stats["n_errors"]}'
      f' ({stats["n_confident_errors"]/stats["n_errors"]:.0%})')

Средняя уверенность на ВЕРНЫХ:    0.930
Средняя уверенность на ОШИБОЧНЫХ: 0.778

Ошибок с уверенностью > 0.9: 14 из 53 (26%)


In [6]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 4.5))
bins = [0.5, 0.6, 0.7, 0.8, 0.9, 0.95, 0.99, 1.0]

ax.hist([df[~df['is_error']]['confidence'], df[df['is_error']]['confidence']],
        bins=bins, label=['Верные', 'Ошибочные'], color=['#4c9f70', '#c94c4c'])
ax.set_xlabel('Уверенность модели')
ax.set_ylabel('Количество предсказаний')
ax.set_title('Распределение уверенности: верные vs ошибочные')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

C:\Users\Vsevolod\AppData\Local\Temp\ipykernel_4332\3335554824.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Находка дня.** На ошибках модель заметно менее уверена, чем на верных ответах: средняя уверенность 0.778 против 0.930. То есть уверенность кое-что говорит о риске — но далеко не всё.

Практическое следствие: **порог по уверенности спасает лишь отчасти**. Даже среди ошибок 14 из 53 (26%) сделаны с уверенностью выше 0.9 — фильтром «доверяем только предсказаниям с p > 0.9» их не отсечь, а сам фильтр вдобавок отбросит часть верных ответов. Это та самая проблема калибровки, которую мы отметили в Дне 6: три эпохи CrossEntropy тянут логиты в крайности, поэтому софтмакс-вероятность нельзя читать как честную оценку риска без калибровки.

### Трудные примеры: где ошиблись все три модели

In [7]:
from error_analysis import find_hard_examples

clf_base = load_baseline()
tok_enc, encoder = load_tokenizer(), load_model()
y_pred_base = [p['prediction'] for p in predict_baseline(test_texts, clf_base, tok_enc, encoder)]

clf_tfidf, vectorizer = load_tfidf()
y_pred_tfidf = [p['prediction'] for p in predict_tfidf(test_texts, clf_tfidf, vectorizer)]

hard = find_hard_examples(df, {'baseline': y_pred_base, 'tfidf': y_pred_tfidf})

print(f'Ошиблись ВСЕ три модели: {len(hard)} примеров\n')
LABELS = {0: 'negative', 1: 'positive'}
for _, row in hard.head(10).iterrows():
    print(f'  [{LABELS[row["true_label"]]:8s}] {row["text"]}')

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Ошиблись ВСЕ три модели: 24 примеров

  [negative] bargain-basement
  [positive] that accomplishes so much that one viewing ca n't possibly be enough
  [positive] that most frightening of all movies
  [negative] stifling morality tale
  [positive] to watch too many barney videos
  [positive] as an intellectual exercise -- an unpleasant debate that 's been given the drive of a narrative and that 's been acted out -- the believer is nothing less than a provocative piece of work
  [negative] jay russell
  [positive] too good to be bad
  [positive] may be why it 's so successful at lodging itself in the brain
  [positive] has any sting to it


**Вот это и есть настоящий разбор ошибок.** 24 примера провалили все три модели — от TF-IDF до дообученного BERT. Когда ошибаются все, дело обычно не в модели. Разберём типы:

**1. Требуется доменное знание.**
> `that most frightening of all movies` → размечено как **positive**

Для фильма ужасов «самый страшный» — это похвала. Модель видит слово `frightening` с негативной окраской в обычном языке.

**2. Идиомы и ирония.**
> `too good to be bad` → **positive**

Конструкция «слишком X, чтобы быть Y» переворачивает смысл. Здесь одновременно `good` и `bad`, и решает грамматика, а не слова.

**3. Фрагменты без тональности вообще.**
> `jay russell` → размечено как **negative**

Это просто **имя режиссёра**. В самом фрагменте тональности нет никакой — метка унаследована из контекста полной рецензии, которого модель не видит.

Причина в устройстве датасета: SST-2 собран из **фраз-фрагментов** полных рецензий (Stanford Sentiment Treebank размечает поддеревья синтаксического разбора). Часть фрагментов в отрыве от контекста не несёт тональности, но метку от родительской фразы сохраняет.

Отсюда вывод: **часть из этих 53 ошибок неустранима**. Это не потолок модели, а потолок разметки. Гнаться за F1 = 1.0 на SST-2 бессмысленно.

### Сохранение анализа

In [8]:
from error_analysis import save_analysis, build_observations

# Наблюдения формируются из посчитанной статистики (build_observations),
# а не пишутся руками — поэтому при повторном запуске на другой версии
# модели выводы автоматически пересчитываются и не устаревают.
observations = build_observations(stats, df, hard_examples=hard)

# n_examples=None — выписываем в отчёт ВСЕ ошибочные примеры (все FP и FN),
# а не только первые пять.
save_analysis(df, stats, 'error_analysis.txt', n_examples=None,
              hard_examples=hard, observations=observations)
print(open('error_analysis.txt', encoding='utf-8').read())

=== АНАЛИЗ ОШИБОК ===

Всего примеров: 600
Всего ошибок: 53 (8.8%)
False Positives (сказали positive, было negative): 24
False Negatives (сказали negative, было positive): 29

--- Длина текстов ---
Средняя длина ошибочных текстов: 53 симв. (10.0 слов)
Средняя длина всех текстов:      52 симв. (9.3 слов)

--- Уверенность модели ---
Средняя уверенность на верных предсказаниях:    0.930
Средняя уверенность на ошибочных предсказаниях: 0.778
Ошибок с уверенностью > 0.9: 14 из 53

=== ПРИМЕРЫ FALSE POSITIVES (сказали good, а было bad) ===  [24 из 24]

Текст: there is plenty of room for editing , and
Истинный: negative, Предсказан: positive, уверенность 0.778

Текст: bargain-basement
Истинный: negative, Предсказан: positive, уверенность 0.747

Текст: fit all of pootie tang in between its punchlines
Истинный: negative, Предсказан: positive, уверенность 0.886

Текст: stifling morality tale
Истинный: negative, Предсказан: positive, уверенность 0.931

Текст: restage the whole thing
Истинный: nega

## Задача 3: Демо-приложение (Gradio)

Готово в [`app.py`](app.py), логика инференса — в [`predictor.py`](predictor.py) (общая с FastAPI).

> ⚠️ **Баг в шаблоне задания.** Там было:
> ```python
> label_map = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}
> ```
> Это карта для **трёх** классов, а наша модель обучена на SST-2 и различает **два**. С этой картой класс 1 (positive) подписывался бы как «Neutral» — то есть все позитивные отзывы демо называло бы нейтральными. Правильно:
> ```python
> LABEL_MAP = {0: 'Negative', 1: 'Positive'}
> ```

In [9]:
from predictor import predict

for text in [
    "This movie was absolutely fantastic!",
    "Terrible, waste of my time.",
    "It was okay, nothing special.",
]:
    r = predict(text)
    print(f'{r["label"]:8s} ({r["confidence"]:.1%})  <- {text}')

Positive (98.6%)  <- This movie was absolutely fantastic!
Negative (98.6%)  <- Terrible, waste of my time.
Negative (74.1%)  <- It was okay, nothing special.


Запуск приложения:

```bash
python app.py
```

Затем откройте http://127.0.0.1:7860 — проверено, интерфейс отвечает и на фразу «This movie was absolutely fantastic!» выдаёт Positive 99%.

## Задача 4: FastAPI

Готово в [`api.py`](api.py). Помимо `/predict` добавлены `/predict_batch` и `/health`.

```bash
uvicorn api:app --reload
```

Проверим эндпоинты, не поднимая сервер, — через `TestClient`:

In [10]:
from fastapi.testclient import TestClient
from api import app as api_app

client = TestClient(api_app)

print('GET /health ->', client.get('/health').json())

r = client.post('/predict', json={'text': 'This movie was great!'})
print(f'\nPOST /predict -> {r.status_code}')
print(r.json())

r = client.post('/predict_batch', json={'texts': ['awful film', 'brilliant work']})
print(f'\nPOST /predict_batch -> {[x["label"] for x in r.json()["results"]]}')

r = client.post('/predict', json={'text': ''})
print(f'\nПустой текст -> {r.status_code} (валидация работает)')

GET /health -> {'status': 'ok'}

POST /predict -> 200
{'label': 'Positive', 'confidence': 0.9827948212623596, 'probabilities': {'Negative': 0.017205139622092247, 'Positive': 0.9827948212623596}}

POST /predict_batch -> ['Negative', 'Positive']

Пустой текст -> 422 (валидация работает)


C:\Users\Vsevolod\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


## Задача 5: README

Готов в [`README.md`](README.md) — со структурой проекта, реальными метриками всех трёх моделей, инструкциями запуска и разделом об ограничениях модели.

## Итоги недели

| День | Что сделали | Результат |
|---|---|---|
| 1 | Токенизация | Текст → subword-токены → ID |
| 2 | Эмбеддинги | CLS-вектор `[1, 768]` из `last_hidden_state` |
| 3 | Attention | 6 слоёв × 12 голов, обнаружен attention sink |
| 4 | Baseline | F1 = 0.8630 на замороженных признаках |
| 5 | Fine-tuning | F1 = 0.9106, обучено 67 млн параметров |
| 6 | Сравнение | TF-IDF 0.72 → заморожен 0.86 → дообучен 0.91 |
| 7 | Анализ ошибок | 53 ошибки, из них 24 неустранимых |

### Три вывода, которые стоит унести

**1. Baseline обязателен.** Без TF-IDF (0.72) казалось бы, что 0.86 у замороженной модели — скромный результат. Без замороженного baseline (0.86) казалось бы, что 0.91 у fine-tuning — огромная победа. Цифра имеет смысл только в сравнении.

**2. Высокая уверенность ≠ правота.** Средняя уверенность на ошибках (0.778) ниже, чем на верных ответах (0.930), — уверенность частично сигналит о риске. Но 14 из 53 ошибок всё равно сделаны с уверенностью > 0.9: софтмакс после обучения на CrossEntropy — это не вероятность в статистическом смысле, и полагаться на неё как на калиброванную оценку риска нельзя.

**3. Часть ошибок — не про модель.** 24 примера провалили все три подхода, включая фрагменты вроде `jay russell` (имя режиссёра, размечено как negative). Прежде чем крутить гиперпараметры ради последних процентов, стоит посмотреть на сами ошибки — иногда там потолок разметки, а не модели.